In [ ]:
# evaluate.py
from torchvision.utils import save_image
from torchvision import transforms
import cv2
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import numpy as np
#import config
#from models import Generator

def calculate_psnr(img1, img2):
    """
    Calculate PSNR between two images.
    Images should be in range [0, 255] and of type uint8.
    """
    if img1.shape != img2.shape:
        # we want to makesure the generated image and the original image have the same dimensions
        raise ValueError("Input images must have the same dimensions")

    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    # find euclidean distance between the two images
    if mse == 0:
        return float('inf')
    # why do this 
    # cannot divide by zero
    # the formulae mse is in denominator

    max_pixel = 255.0
    # set by the images bit
    # why 255.0?
    # because the pixel values are in range [0, 255]
    # how do we know
    # 2^8 = 256 distinct values and they typically range from 0 to 255
    psnr_value = 20 * np.log10(max_pixel / np.sqrt(mse))
    # the formula for psnr
    return psnr_value

def calculate_ssim(img1, img2):
    """
    Calculate SSIM between two images.
    Images should be in range [0, 255] and of type uint8.
    """
    if img1.shape != img2.shape:
        raise ValueError("Input images must have the same dimensions")

    # Convert to grayscale if images are color
    if len(img1.shape) == 3:
        # check if the image is 3 dimension, height, weight and channel, if 2 dimension is greyscale
        img1_gray = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        # if it is color use a function to convert it to grayscale
        img2_gray = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    else:
        img1_gray = img1
        # if it is greyscale, just use the image as it is
        img2_gray = img2


    ssim_value = ssim(img1_gray, img2_gray, data_range=255)
    # data_range is the range of the pixel values
    # for uint8 images, the range is 0 to 255
    # ssim is a measure of similarity between two images
    # it considers luminance, contrast, and structure using a formula
    return ssim_value

def tensor_to_numpy(tensor):
    """
    Convert tensor to numpy array in range [0, 255].
    """
    # Denormalize from [-1, 1] to [0, 1]
    # in the Dataset we convert the image to tensor and for HR and also the generated image there are in that range
    # so we need to convert it back to [0, 1] range
    # let x = [-1,1]
    # x + 1 = [0, 2] and 0.5 * (x + 1) = [0, 1]
    # x = x * 0.5 + 0.5 = 0.5 (x + 1)
    tensor = tensor * 0.5 + 0.5
    # Clamp to [0, 1]
    tensor = torch.clamp(tensor, 0, 1)
    # make sure all tensor values are between 0 and 1
    # if the values are less than 0, set them to 0
    # avoid error why we scale back to [0,255] to display the image
    # as the range is preserved

    # Convert to numpy and scale to [0, 255]
    numpy_img = tensor.squeeze(0).cpu().detach().numpy()
    # remove the batch dimension as torch.randn(batch, channels, height, width) is used to create the tensor
    # move the tensor to CPU and detach it from the computation graph
    # no gradient will be computer for this tensor
    # convert the tensor to numpy array

    numpy_img = np.transpose(numpy_img, (1, 2, 0))  # CHW to HWC
    # change the shape from (channels, height, width) to (height, width, channels)
    # most image library is HWC so we reverse the order
    numpy_img = (numpy_img * 255).astype(np.uint8)
    # scale the pixel values to [0, 255] and convert to uint8
    return numpy_img
    # this is the format that most image library use and we can use it to display the image